In [5]:
from pyspark.sql import functions as F

bronze_path = "Files/bronze/nyc_taxi/yellow/"

df = (
    spark.read
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .parquet(bronze_path)
)

# rename (only if columns exist)
rename_map = {
    "VendorID": "vendor_id",
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "RatecodeID": "ratecode_id",
    "PULocationID": "pu_location_id",
    "DOLocationID": "do_location_id"
}
for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

# Ensure timestamps
if "pickup_datetime" in df.columns:
    df = df.withColumn("pickup_datetime", F.to_timestamp("pickup_datetime"))
if "dropoff_datetime" in df.columns:
    df = df.withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime"))

# year/month (prefer partition columns if they exist, otherwise from pickup_datetime)
if "year" in df.columns and "month" in df.columns:
    df = df.withColumn("year", F.col("year").cast("int")) \
           .withColumn("month", F.col("month").cast("int"))
else:
    df = df.withColumn("year", F.year("pickup_datetime")) \
           .withColumn("month", F.month("pickup_datetime"))

# numeric casts (only if exist)
to_double = ["trip_distance","fare_amount","extra","mta_tax","tip_amount","tolls_amount",
             "improvement_surcharge","total_amount","congestion_surcharge","airport_fee"]
to_int = ["passenger_count","ratecode_id","pu_location_id","do_location_id","payment_type","vendor_id"]

for c in to_double:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))
for c in to_int:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("int"))

# derived metric
df = df.withColumn(
    "trip_duration_min",
    (F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")) / 60.0
)

# basic cleaning filters (keep reasonable rows)
df_silver = df.filter(
    F.col("pickup_datetime").isNotNull() &
    F.col("dropoff_datetime").isNotNull() &
    (F.col("trip_duration_min") >= 0) &
    (F.col("trip_duration_min") <= 24*60) &
    (F.col("trip_distance").isNull() | (F.col("trip_distance") >= 0)) &
    (F.col("total_amount").isNull() | (F.col("total_amount") >= 0))
)

# write Silver table
spark.sql("DROP TABLE IF EXISTS silver_nyc_taxi_yellow")

(df_silver
 .write
 .format("delta")
 .mode("overwrite")
 .partitionBy("year","month")
 .saveAsTable("silver_nyc_taxi_yellow")
)


spark.sql("""
SELECT year, month, COUNT(*) AS rows
FROM silver_nyc_taxi_yellow
GROUP BY year, month
ORDER BY year, month
""").show(50, False)


StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 7, Finished, Available, Finished)

+----+-----+-------+
|year|month|rows   |
+----+-----+-------+
|2002|12   |10     |
|2008|12   |10     |
|2009|1    |18     |
|2023|12   |10     |
|2024|1    |2929042|
|2024|2    |2971435|
|2024|3    |3538164|
|2024|4    |3470450|
|2024|5    |3674793|
|2024|6    |3490348|
|2024|7    |3026467|
|2024|8    |2926218|
|2024|9    |3576848|
|2024|10   |3772298|
|2024|11   |3584733|
|2024|12   |3597758|
|2025|1    |3412027|
|2025|2    |2      |
|2025|3    |2      |
|2026|6    |2      |
+----+-----+-------+



In [8]:
from pyspark.sql import functions as F

# Read Bronze table
df = spark.table("bronze_ecb_fx")

# Clean + types
df_silver = (
    df.select(
        F.to_date("date").alias("date"),
        F.col("usd_per_eur").cast("double").alias("usd_per_eur"),
        F.col("eur_per_usd").cast("double").alias("eur_per_usd")
    )
    .filter(F.col("date").isNotNull())
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
)

# Write Silver table
spark.sql("DROP TABLE IF EXISTS silver_ecb_fx")

(df_silver
 .write
 .format("delta")
 .mode("overwrite")
 .partitionBy("year","month")
 .saveAsTable("silver_ecb_fx")
)


StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 10, Finished, Available, Finished)

In [9]:
spark.sql("SELECT year, month, COUNT(*) rows FROM silver_ecb_fx GROUP BY year, month ORDER BY year, month").show(50, False)

StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 11, Finished, Available, Finished)

+----+-----+----+
|year|month|rows|
+----+-----+----+
|2025|1    |22  |
|2025|2    |20  |
|2025|3    |21  |
|2025|4    |20  |
|2025|5    |21  |
|2025|6    |21  |
|2025|7    |23  |
|2025|8    |21  |
|2025|9    |22  |
|2025|10   |23  |
|2025|11   |20  |
|2025|12   |20  |
+----+-----+----+



In [10]:
from pyspark.sql import functions as F

# 1) Read bronze (try without schema, then with dbo.)
try:
    df = spark.table("bronze_worldbank_gdp")
    src_name = "bronze_worldbank_gdp"
except:
    df = spark.table("dbo.bronze_worldbank_gdp")
    src_name = "dbo.bronze_worldbank_gdp"

print("Using source table:", src_name)
print("Columns:", df.columns)
display(df.limit(5))

# 2) If indicator columns are missing, set them as constants
cols = set(df.columns)

indicator_code_expr = (
    F.col("indicator_code").cast("string")
    if "indicator_code" in cols
    else F.lit("NY.GDP.MKTP.CD")
)

indicator_name_expr = (
    F.col("indicator_name").cast("string")
    if "indicator_name" in cols
    else F.lit("GDP (current US$)")
)

df_silver = (
    df.select(
        indicator_code_expr.alias("indicator_code"),
        indicator_name_expr.alias("indicator_name"),
        F.col("country_name").cast("string").alias("country_name"),
        F.col("countryiso3code").cast("string").alias("country_iso3"),
        F.col("year").cast("int").alias("year"),
        F.col("value").cast("double").alias("gdp_usd"),
    )
    .filter(F.col("year").isNotNull() & F.col("gdp_usd").isNotNull())
)

# 3) Write silver table
spark.sql("DROP TABLE IF EXISTS silver_worldbank_gdp")
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_worldbank_gdp")

display(spark.table("silver_worldbank_gdp").limit(10))


StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 12, Finished, Available, Finished)

Using source table: bronze_worldbank_gdp
Columns: ['country_name', 'countryiso3code', 'year', 'value']


SynapseWidget(Synapse.DataFrame, fee6f166-c8e4-4dbc-aa52-69c908f6129f)

SynapseWidget(Synapse.DataFrame, b5688086-53f7-4b7e-a008-316817a38447)

In [11]:
spark.sql("SELECT country_iso3, MIN(year) min_year, MAX(year) max_year, COUNT(*) rows FROM silver_worldbank_gdp GROUP BY country_iso3 ORDER BY rows DESC").show(50, False)

StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 13, Finished, Available, Finished)

+------------+--------+--------+----+
|country_iso3|min_year|max_year|rows|
+------------+--------+--------+----+
|            |1960    |2024    |253 |
|VCT         |1960    |2024    |65  |
|MEA         |1960    |2024    |65  |
|HTI         |1960    |2024    |65  |
|FIN         |1960    |2024    |65  |
|BRB         |1960    |2024    |65  |
|BHS         |1960    |2024    |65  |
|TEA         |1960    |2024    |65  |
|GBR         |1960    |2024    |65  |
|ZMB         |1960    |2024    |65  |
|JAM         |1960    |2024    |65  |
|MIC         |1960    |2024    |65  |
|IDA         |1960    |2024    |65  |
|BRA         |1960    |2024    |65  |
|FRA         |1960    |2024    |65  |
|PRY         |1960    |2024    |65  |
|EAR         |1960    |2024    |65  |
|SSA         |1960    |2024    |65  |
|COD         |1960    |2024    |65  |
|URY         |1960    |2024    |65  |
|CRI         |1960    |2024    |65  |
|OED         |1960    |2024    |65  |
|BMU         |1960    |2024    |65  |
|LBY        

In [12]:
from pyspark.sql import functions as F

# Read bronze ECB FX (try both names)
try:
    df = spark.table("bronze_ecb_fx")
    src_name = "bronze_ecb_fx"
except:
    df = spark.table("dbo.bronze_ecb_fx")
    src_name = "dbo.bronze_ecb_fx"

print("Using source table:", src_name)
print("Columns:", df.columns)
display(df.limit(5))

# Clean + types
df_silver = (
    df.select(
        F.col("date").cast("date").alias("date"),
        F.col("usd_per_eur").cast("double").alias("usd_per_eur"),
        F.col("eur_per_usd").cast("double").alias("eur_per_usd"),
    )
    .filter(F.col("date").isNotNull())
)

# Write silver table
spark.sql("DROP TABLE IF EXISTS silver_ecb_fx")
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_ecb_fx")

display(spark.table("silver_ecb_fx").limit(10))


StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 14, Finished, Available, Finished)

Using source table: bronze_ecb_fx
Columns: ['date', 'usd_per_eur', 'eur_per_usd']


SynapseWidget(Synapse.DataFrame, d5392969-a3ac-48b2-affc-394a7054ee12)

SynapseWidget(Synapse.DataFrame, 799ef43f-5536-4a37-8633-620a68bb1744)

In [13]:
from pyspark.sql import functions as F

# 1) Read BRONZE files (yellow taxi) from Lakehouse Files
path = "Files/bronze/nyc_taxi/yellow/"
df = spark.read.parquet(path)

print("Loaded rows:", df.count())
print("Columns:", df.columns)
display(df.limit(5))

# Helper: safe column getter (won't crash if a column is missing)
def c(name):
    return F.col(name) if name in df.columns else F.lit(None)

# 2) Build SILVER (clean names + types)
df_silver = (
    df.select(
        c("VendorID").cast("int").alias("vendor_id"),
        c("tpep_pickup_datetime").cast("timestamp").alias("pickup_ts"),
        c("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_ts"),
        c("passenger_count").cast("int").alias("passenger_count"),
        c("trip_distance").cast("double").alias("trip_distance"),
        c("PULocationID").cast("int").alias("pu_location_id"),
        c("DOLocationID").cast("int").alias("do_location_id"),
        c("payment_type").cast("int").alias("payment_type"),
        c("fare_amount").cast("double").alias("fare_amount"),
        c("tip_amount").cast("double").alias("tip_amount"),
        c("total_amount").cast("double").alias("total_amount"),
        c("year").cast("int").alias("year"),
        c("month").cast("int").alias("month"),
    )
    .filter(F.col("pickup_ts").isNotNull())
)

# 3) Write SILVER table
spark.sql("DROP TABLE IF EXISTS silver_nyc_taxi_yellow")
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_nyc_taxi_yellow")

# 4) Quick check
display(spark.table("silver_nyc_taxi_yellow").limit(10))
print("SILVER rows:", spark.table("silver_nyc_taxi_yellow").count())


StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 15, Finished, Available, Finished)

Loaded rows: 41169720
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'year', 'month']


SynapseWidget(Synapse.DataFrame, c3a00f92-d3a9-48f3-93b5-046f995592b8)

SynapseWidget(Synapse.DataFrame, 95cd8793-ceb0-4bb0-8cf1-95da265696d8)

SILVER rows: 41169720


In [14]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_nyc_taxi_yellow")

StatementMeta(, 6e6258f4-3e61-4d76-a5d4-4dd4a0ac5b9d, 16, Finished, Available, Finished)